# AI-Powered Student Performance & Placement Analytics
**Author:** Anushka  
**Program:** IBM SkillsBuild Data Analytics with AI Academic Internship – BharatCares / AICTE  
**Dataset:** placementdata.csv — 10,000 students, 12 features  

## Objective
Predict whether a student will be placed or not placed, using academic scores, aptitude scores, soft skills ratings, and extracurricular engagement features. Apply and compare six classification models to identify the most effective approach.

## Step 1 — Dataset Understanding

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("placementdata.csv")

print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns\n")
print("Column names and data types:")
print(df_raw.dtypes)
print(f"\nMissing values:\n{df_raw.isnull().sum()}")
print(f"\nDuplicate rows: {df_raw.duplicated().sum()}")
print(f"\nTarget column value counts:\n{df_raw['PlacementStatus'].value_counts()}")
print(f"\nDescriptive statistics:")
display(df_raw.describe().round(2))

## Step 2 — Exploratory Data Analysis

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.gridspec import GridSpec

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
})

PLACED_COLOR    = "#2563EB"
NOTPLACED_COLOR = "#EF4444"
PALETTE         = {"Placed": PLACED_COLOR, "NotPlaced": NOTPLACED_COLOR}

df_eda = pd.read_csv("placementdata.csv")
df_eda = df_eda.rename(columns={"Workshops/Certifications": "Workshops_Certifications"})
print("EDA dataset loaded:", df_eda.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.suptitle("Figure 1: PlacementStatus Distribution", fontsize=14, fontweight="bold")

counts = df_eda["PlacementStatus"].value_counts()
pcts   = df_eda["PlacementStatus"].value_counts(normalize=True) * 100

bars = axes[0].bar(counts.index, counts.values,
                   color=[PALETTE[c] for c in counts.index],
                   edgecolor="white", width=0.5)
for bar, val, pct in zip(bars, counts.values, pcts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 80,
                 f"{val:,}\n({pct:.1f}%)", ha="center", va="bottom",
                 fontsize=11, fontweight="bold")
axes[0].set_title("Count of Students per Class")
axes[0].set_ylabel("Number of Students")
axes[0].set_xlabel("Placement Status")
axes[0].set_ylim(0, 7000)

axes[1].pie(counts.values, labels=counts.index, autopct="%1.1f%%",
            colors=[PALETTE[c] for c in counts.index],
            startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 2},
            textprops={"fontsize": 12})
axes[1].set_title("Proportion of Each Class")

plt.tight_layout()
plt.show()
print("Interpretation: 58.03% Not Placed, 41.97% Placed. Mild class imbalance -- manageable with stratified split.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Figure 2: CGPA Distribution by PlacementStatus", fontsize=14, fontweight="bold")

for status, color in PALETTE.items():
    subset = df_eda[df_eda["PlacementStatus"] == status]["CGPA"]
    axes[0].hist(subset, bins=28, alpha=0.55, color=color, edgecolor="white", label=status)
    subset.plot.kde(ax=axes[0], color=color, linewidth=2.2)
axes[0].set_title("CGPA Histogram + Density")
axes[0].set_xlabel("CGPA")
axes[0].set_ylabel("Count / Density")
axes[0].legend(title="Placement Status")

sns.boxplot(data=df_eda, x="PlacementStatus", y="CGPA", hue="PlacementStatus",
            palette=PALETTE, width=0.45, linewidth=1.5, ax=axes[1], legend=False)
means = df_eda.groupby("PlacementStatus")["CGPA"].mean()
for i, (status, mean_val) in enumerate(means.items()):
    axes[1].text(i, mean_val + 0.04, f"mu={mean_val:.2f}",
                 ha="center", fontsize=10, fontweight="bold", color=PALETTE[status])
axes[1].set_title("CGPA Box Plot")
axes[1].set_xlabel("Placement Status")
axes[1].set_ylabel("CGPA")
plt.tight_layout()
plt.show()
print("Interpretation: Placed students have a higher median CGPA (~8.0 vs ~7.5). Clear inflection at CGPA >= 8.0.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Figure 3: SSC & HSC Marks vs PlacementStatus", fontsize=14, fontweight="bold")

for ax, col, title in zip(axes, ["SSC_Marks", "HSC_Marks"], ["SSC Marks (Class 10)", "HSC Marks (Class 12)"]):
    sns.violinplot(data=df_eda, x="PlacementStatus", y=col, hue="PlacementStatus",
                   palette=PALETTE, inner="quartile", linewidth=1.4, ax=ax, legend=False)
    means = df_eda.groupby("PlacementStatus")[col].mean()
    for i, (status, mean_val) in enumerate(means.items()):
        ax.scatter(i, mean_val, s=60, zorder=5, color=PALETTE[status], edgecolor="white", linewidth=1.5)
        ax.text(i, mean_val + 1.2, f"mu={mean_val:.1f}", ha="center", fontsize=10,
                fontweight="bold", color=PALETTE[status])
    ax.set_title(title)
    ax.set_xlabel("Placement Status")
    ax.set_ylabel("Marks (%)")

plt.tight_layout()
plt.show()
print("Interpretation: Placed students score higher in both SSC (mean 73.5 vs 66.1) and HSC (mean 79.4 vs 71.1).")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Figure 4: Aptitude Test Score vs PlacementStatus", fontsize=14, fontweight="bold")

for status, color in PALETTE.items():
    subset = df_eda[df_eda["PlacementStatus"] == status]["AptitudeTestScore"]
    axes[0].hist(subset, bins=31, alpha=0.5, color=color, edgecolor="white", label=status)
    subset.plot.kde(ax=axes[0], color=color, linewidth=2.2)
axes[0].axvline(85, color="black", linestyle="--", linewidth=1.3, alpha=0.7)
axes[0].text(85.5, axes[0].get_ylim()[1] * 0.9, "Score=85", fontsize=9, color="black")
axes[0].set_title("Aptitude Score Histogram + Density")
axes[0].set_xlabel("Aptitude Test Score")
axes[0].set_ylabel("Count / Density")
axes[0].legend(title="Placement Status")

sns.boxplot(data=df_eda, x="PlacementStatus", y="AptitudeTestScore", hue="PlacementStatus",
            palette=PALETTE, width=0.45, linewidth=1.5, ax=axes[1], legend=False)
means = df_eda.groupby("PlacementStatus")["AptitudeTestScore"].mean()
for i, (status, mean_val) in enumerate(means.items()):
    axes[1].text(i, mean_val + 0.5, f"mu={mean_val:.1f}", ha="center", fontsize=10,
                 fontweight="bold", color=PALETTE[status])
axes[1].set_title("Aptitude Score Box Plot")
axes[1].set_xlabel("Placement Status")
axes[1].set_ylabel("Aptitude Test Score")
plt.tight_layout()
plt.show()
print("Interpretation: AptitudeTestScore is the strongest numerical predictor (r=0.52). Mean 84.5 (Placed) vs 75.8 (Not Placed).")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Figure 5: Soft Skills Rating vs PlacementStatus", fontsize=14, fontweight="bold")

sns.boxplot(data=df_eda, x="PlacementStatus", y="SoftSkillsRating", hue="PlacementStatus",
            palette=PALETTE, width=0.4, linewidth=1.5, ax=axes[0], fliersize=0, legend=False)
sns.stripplot(data=df_eda.sample(600, random_state=42), x="PlacementStatus", y="SoftSkillsRating",
              hue="PlacementStatus", palette=PALETTE, alpha=0.25, size=3.5, jitter=True,
              ax=axes[0], legend=False)
means = df_eda.groupby("PlacementStatus")["SoftSkillsRating"].mean()
for i, (status, mean_val) in enumerate(means.items()):
    axes[0].text(i, mean_val + 0.04, f"mu={mean_val:.2f}", ha="center", fontsize=10,
                 fontweight="bold", color=PALETTE[status])
axes[0].set_title("Box + Strip Plot")
axes[0].set_xlabel("Placement Status")
axes[0].set_ylabel("Soft Skills Rating (3.0-4.8)")

rating_dist = df_eda.groupby(["SoftSkillsRating", "PlacementStatus"]).size().unstack(fill_value=0)
rating_dist.plot(kind="bar", ax=axes[1], color=[NOTPLACED_COLOR, PLACED_COLOR],
                 edgecolor="white", width=0.8)
axes[1].set_title("Rating Distribution by Class")
axes[1].set_xlabel("Soft Skills Rating")
axes[1].set_ylabel("Number of Students")
axes[1].legend(title="Placement Status")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()
print("Interpretation: Placed students average 4.53 vs 4.17. Ratings >= 4.5 are strongly associated with placement.")

In [ ]:
fig = plt.figure(figsize=(16, 10))
fig.suptitle("Figure 6: Activity & Engagement Features vs PlacementStatus", fontsize=14, fontweight="bold")
gs = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

plot_specs = [
    ("Internships",               "Number of Internships",      gs[0, 0]),
    ("Projects",                  "Number of Projects",         gs[0, 1]),
    ("Workshops_Certifications",  "Workshops / Certifications", gs[0, 2]),
    ("ExtracurricularActivities", "Extracurricular Activities", gs[1, 0]),
    ("PlacementTraining",         "Placement Training",         gs[1, 1]),
]
for col, title, pos in plot_specs:
    ax = fig.add_subplot(pos)
    ct = df_eda.groupby([col, "PlacementStatus"]).size().unstack(fill_value=0)
    if "NotPlaced" not in ct.columns: ct["NotPlaced"] = 0
    if "Placed"    not in ct.columns: ct["Placed"]    = 0
    ct = ct[["NotPlaced", "Placed"]]
    ct.plot(kind="bar", ax=ax, color=[NOTPLACED_COLOR, PLACED_COLOR], edgecolor="white", width=0.65)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("")
    ax.set_ylabel("Students")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(title="", fontsize=9,
              handles=[mpatches.Patch(color=NOTPLACED_COLOR, label="NotPlaced"),
                       mpatches.Patch(color=PLACED_COLOR,    label="Placed")])
plt.tight_layout()
plt.show()
print("Interpretation: 2 internships->70% placed. 3 projects->72%. Extracurricular Yes->62% vs No->14%. Training Yes->52% vs No->16%.")

In [ ]:
df_enc = df_eda.copy()
df_enc["PlacementStatus"]           = (df_enc["PlacementStatus"] == "Placed").astype(int)
df_enc["ExtracurricularActivities"] = (df_enc["ExtracurricularActivities"] == "Yes").astype(int)
df_enc["PlacementTraining"]         = (df_enc["PlacementTraining"] == "Yes").astype(int)

corr_cols = ["CGPA", "SSC_Marks", "HSC_Marks", "AptitudeTestScore", "SoftSkillsRating",
             "Internships", "Projects", "Workshops_Certifications",
             "ExtracurricularActivities", "PlacementTraining", "PlacementStatus"]
corr_matrix = df_enc[corr_cols].corr()
display_labels = ["CGPA", "SSC Marks", "HSC Marks", "Aptitude Score", "Soft Skills",
                  "Internships", "Projects", "Workshops", "Extracurricular",
                  "Placement Training", "Placement Status"]
corr_matrix.columns = display_labels
corr_matrix.index   = display_labels

mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f",
            cmap=sns.diverging_palette(220, 10, as_cmap=True),
            vmin=-1, vmax=1, center=0, linewidths=0.5, linecolor="#e5e7eb",
            annot_kws={"size": 9}, square=True, ax=ax,
            cbar_kws={"shrink": 0.75, "label": "Pearson r"})
ax.set_title("Figure 7: Feature-Target Correlation Heatmap", fontsize=13, fontweight="bold", pad=10)
ax.tick_params(axis="x", rotation=45, labelsize=10)
ax.tick_params(axis="y", rotation=0,  labelsize=10)
plt.tight_layout()
plt.show()
print("Interpretation: AptitudeTestScore (r=0.52), HSC_Marks (r=0.51), ExtracurricularActivities (r=0.48) are the top predictors.")

## Step 3 — Data Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("placementdata.csv")
df = df.drop(columns=["StudentID"])
df = df.rename(columns={"Workshops/Certifications": "Workshops_Certifications"})
df["ExtracurricularActivities"] = df["ExtracurricularActivities"].map({"Yes": 1, "No": 0})
df["PlacementTraining"]         = df["PlacementTraining"].map({"Yes": 1, "No": 0})
df["PlacementStatus"]           = df["PlacementStatus"].map({"Placed": 1, "NotPlaced": 0})

FEATURE_COLS = [
    "CGPA", "Internships", "Projects", "Workshops_Certifications",
    "AptitudeTestScore", "SoftSkillsRating",
    "ExtracurricularActivities", "PlacementTraining",
    "SSC_Marks", "HSC_Marks",
]
TARGET_COL = "PlacementStatus"

X = df[FEATURE_COLS]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURE_COLS, index=X_train.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),      columns=FEATURE_COLS, index=X_test.index)

print("Preprocessing complete.")
print(f"  Full dataset : {X.shape[0]:,} samples x {X.shape[1]} features")
print(f"  Train set    : {X_train.shape[0]:,} samples  ({y_train.mean()*100:.2f}% Placed)")
print(f"  Test set     : {X_test.shape[0]:,} samples  ({y_test.mean()*100:.2f}% Placed)")
print(f"\nFeature columns  : {FEATURE_COLS}")
print(f"Target column    : {TARGET_COL}  (Placed=1, NotPlaced=0)")
display(df.head(3))

## Step 4 — Feature Analysis

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance

COLORS_6 = ["#2563EB", "#EF4444", "#10B981", "#F59E0B", "#7C3AED", "#EC4899"]
NEUTRAL   = "#6B7280"
ACCENT    = "#2563EB"

rf_feat = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_feat.fit(X_train, y_train)

gb_feat = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42)
gb_feat.fit(X_train, y_train)

perm = permutation_importance(rf_feat, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1)

rf_imp   = pd.Series(rf_feat.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
gb_imp   = pd.Series(gb_feat.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
perm_df  = pd.DataFrame({"Feature": FEATURE_COLS, "Mean": perm.importances_mean,
                          "Std": perm.importances_std}).sort_values("Mean", ascending=False).reset_index(drop=True)

combined = pd.DataFrame(index=FEATURE_COLS)
combined["RF"]   = rf_feat.feature_importances_
combined["GB"]   = gb_feat.feature_importances_
combined["Perm"] = perm.importances_mean
combined["RF_Rank"]   = combined["RF"].rank(ascending=False).astype(int)
combined["GB_Rank"]   = combined["GB"].rank(ascending=False).astype(int)
combined["Perm_Rank"] = combined["Perm"].rank(ascending=False).astype(int)
combined["Avg_Rank"]  = (combined["RF_Rank"] + combined["GB_Rank"] + combined["Perm_Rank"]) / 3
combined = combined.sort_values("Avg_Rank")

print("Feature Importance -- Consolidated Ranking:")
print(f"{'Feature':<30} {'RF Rank':>8} {'GB Rank':>8} {'Perm Rank':>10} {'Avg Rank':>10}")
print("-" * 68)
for feat, row in combined.iterrows():
    print(f"{feat:<30} {int(row['RF_Rank']):>8} {int(row['GB_Rank']):>8} {int(row['Perm_Rank']):>10} {row['Avg_Rank']:>10.1f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle("Figure 8: Feature Importance Analysis", fontsize=14, fontweight="bold")

colors_rf = [ACCENT if i < 3 else NEUTRAL for i in range(len(rf_imp))]
bars = axes[0].barh(rf_imp.index[::-1], rf_imp.values[::-1],
                     color=colors_rf[::-1], edgecolor="white", height=0.65)
for bar, val in zip(bars, rf_imp.values[::-1]):
    axes[0].text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                 f"{val:.3f}", va="center", ha="left", fontsize=9)
axes[0].set_title("A) Random Forest (MDI)")
axes[0].set_xlabel("Importance Score")
axes[0].set_xlim(0, rf_imp.max() * 1.22)

perm_plot = perm_df.sort_values("Mean")
colors_pm = [ACCENT if i >= len(perm_plot)-3 else NEUTRAL for i in range(len(perm_plot))]
axes[1].barh(perm_plot["Feature"], perm_plot["Mean"], xerr=perm_plot["Std"],
             color=colors_pm, edgecolor="white", height=0.65,
             error_kw={"ecolor": "#9CA3AF", "capsize": 3, "linewidth": 1.2})
axes[1].set_title("B) Permutation Importance")
axes[1].set_xlabel("Mean Accuracy Drop")

combined_s = combined.sort_values("Avg_Rank")
y_pos = range(len(combined_s))
axes[2].scatter(combined_s["RF_Rank"],   y_pos, s=60, label="RF",   color=ACCENT)
axes[2].scatter(combined_s["GB_Rank"],   y_pos, s=60, label="GB",   color="#7C3AED")
axes[2].scatter(combined_s["Perm_Rank"], y_pos, s=60, label="Perm", color="#10B981")
axes[2].scatter(combined_s["Avg_Rank"],  y_pos, s=90, label="Avg",  color="#F59E0B", marker="D")
for i, (_, row) in enumerate(combined_s.iterrows()):
    ranks = [row["RF_Rank"], row["GB_Rank"], row["Perm_Rank"]]
    axes[2].hlines(i, min(ranks), max(ranks), colors="#D1D5DB", linewidth=1.5)
axes[2].set_yticks(list(y_pos))
axes[2].set_yticklabels(combined_s.index, fontsize=9)
axes[2].set_xlabel("Rank (1 = most important)")
axes[2].set_title("C) Rank Comparison")
axes[2].invert_xaxis()
axes[2].legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
corr_feat = X[FEATURE_COLS].corr()
fig, ax = plt.subplots(figsize=(10, 8))
mask_f = np.triu(np.ones_like(corr_feat, dtype=bool), k=1)
sns.heatmap(corr_feat, mask=mask_f, annot=True, fmt=".2f",
            cmap=sns.diverging_palette(220, 10, as_cmap=True),
            vmin=-1, vmax=1, center=0, linewidths=0.5, linecolor="#e5e7eb",
            annot_kws={"size": 9}, square=True, ax=ax,
            cbar_kws={"shrink": 0.75, "label": "Pearson r"})
ax.set_title("Figure 9: Feature-Feature Correlation Heatmap\n(No pair exceeds |r|=0.70 -- no multicollinearity)", 
             fontsize=12, fontweight="bold", pad=10)
ax.tick_params(axis="x", rotation=45, labelsize=9)
ax.tick_params(axis="y", rotation=0, labelsize=9)
plt.tight_layout()
plt.show()
print("VIF analysis confirmed all values < 2.0. No multicollinearity. All 10 features retained.")

## Step 5 — Machine Learning Model Building

In [ ]:
import time
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

MODELS = [
    ("Logistic Regression", LogisticRegression(max_iter=1000, random_state=42, solver="lbfgs", C=1.0), True),
    ("Decision Tree",       DecisionTreeClassifier(min_samples_leaf=5, random_state=42), False),
    ("Random Forest",       RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1), False),
    ("Gradient Boosting",   GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42), False),
    ("SVM",                 CalibratedClassifierCV(SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42), ensemble=False), True),
    ("KNN",                 KNeighborsClassifier(n_neighbors=11, metric="minkowski", p=2), True),
]

results = []
print(f"{'Model':<25} {'Data':<10} {'Train Acc':>10} {'Test Acc':>10} {'Time (s)':>10}")
print("-" * 70)

for name, model, use_scaled in MODELS:
    X_tr = X_train_scaled if use_scaled else X_train
    X_te = X_test_scaled  if use_scaled else X_test
    t0 = time.time()
    model.fit(X_tr, y_train)
    t = time.time() - t0
    train_acc = accuracy_score(y_train, model.predict(X_tr))
    test_acc  = accuracy_score(y_test,  model.predict(X_te))
    print(f"{name:<25} {'Scaled' if use_scaled else 'Unscaled':<10} {train_acc:>10.4f} {test_acc:>10.4f} {t:>10.3f}")
    results.append({"Model": name, "model_obj": model, "use_scaled": use_scaled,
                    "Train_Acc": round(train_acc,4), "Test_Acc": round(test_acc,4)})

trained_models = {r["Model"]: r for r in results}

## Step 6 — Model Evaluation

In [ ]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, confusion_matrix, classification_report
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

MODEL_ORDER   = ["Logistic Regression", "Decision Tree", "Random Forest",
                 "Gradient Boosting", "SVM", "KNN"]
SCALED_MODELS = {"Logistic Regression", "SVM", "KNN"}
COLORS_6      = ["#2563EB", "#EF4444", "#10B981", "#F59E0B", "#7C3AED", "#EC4899"]

In [ ]:
eval_records = []

for name in MODEL_ORDER:
    r      = trained_models[name]
    model  = r["model_obj"]
    X_te   = X_test_scaled if name in SCALED_MODELS else X_test
    X_tr   = X_train_scaled if name in SCALED_MODELS else X_train
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    eval_records.append({
        "Model":     name,
        "Train_Acc": accuracy_score(y_train, model.predict(X_tr)),
        "Test_Acc":  accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall":    recall_score(y_test, y_pred, zero_division=0),
        "F1_Score":  f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC":   roc_auc_score(y_test, y_prob),
        "y_pred":    y_pred,
        "y_prob":    y_prob,
    })

eval_df = pd.DataFrame(eval_records)
eval_df["Overfit_Gap"] = eval_df["Train_Acc"] - eval_df["Test_Acc"]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("Figure 10: Confusion Matrices (Test Set, n=2,000)", fontsize=14, fontweight="bold")

for ax, (name, color) in zip(axes.flat, zip(MODEL_ORDER, COLORS_6)):
    row = eval_df[eval_df["Model"] == name].iloc[0]
    cm  = confusion_matrix(y_test, row["y_pred"])
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    annot  = np.array([[f"{cm[i,j]}\n({cm_pct[i,j]:.1f}%)" for j in range(2)] for i in range(2)])
    sns.heatmap(cm, annot=annot, fmt="", ax=ax,
                cmap=sns.light_palette(color, as_cmap=True),
                linewidths=1.5, linecolor="white",
                xticklabels=["NotPlaced","Placed"],
                yticklabels=["NotPlaced","Placed"],
                cbar=False, annot_kws={"size": 11})
    ax.set_title(f"{name}\nAcc={row['Test_Acc']:.4f}  F1={row['F1_Score']:.4f}", fontsize=11)
    ax.set_xlabel("Predicted", fontsize=9)
    ax.set_ylabel("Actual", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
ax.plot([0,1],[0,1],"k--",linewidth=1,alpha=0.6,label="Random classifier (AUC=0.50)")
for name, color in zip(MODEL_ORDER, COLORS_6):
    row = eval_df[eval_df["Model"]==name].iloc[0]
    fpr, tpr, _ = roc_curve(y_test, row["y_prob"])
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f"{name}  (AUC = {row['ROC_AUC']:.4f})")
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title("Figure 11: ROC Curves -- All Six Models\n(Test Set, n=2,000)", fontsize=12, fontweight="bold")
ax.legend(loc="lower right", fontsize=10)
ax.set_xlim(0,1); ax.set_ylim(0,1.02)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_full_cv = df[FEATURE_COLS]
y_full_cv = df[TARGET_COL]

cv_records = []
print("5-Fold Stratified Cross-Validation (full dataset, n=10,000):\n")
print(f"{'Model':<22} {'CV Acc Mean':>12} {'CV Acc Std':>11} {'CV F1 Mean':>11} {'CV F1 Std':>10}")
print("-" * 68)

for name in MODEL_ORDER:
    r = trained_models[name]
    base_model = r["model_obj"]
    if name in SCALED_MODELS:
        pipeline = Pipeline([("scaler", StandardScaler()), ("clf", base_model)])
        X_cv = X_full_cv
    else:
        pipeline = base_model
        X_cv = X_full_cv
    cv_res = cross_validate(pipeline, X_cv, y_full_cv, cv=skf,
                             scoring={"accuracy":"accuracy","f1":"f1"}, n_jobs=-1)
    acc_mean = cv_res["test_accuracy"].mean()
    acc_std  = cv_res["test_accuracy"].std()
    f1_mean  = cv_res["test_f1"].mean()
    f1_std   = cv_res["test_f1"].std()
    print(f"  {name:<22} {acc_mean:>12.4f} {acc_std:>11.4f} {f1_mean:>11.4f} {f1_std:>10.4f}")
    cv_records.append({"Model": name, "CV_Acc_Mean": round(acc_mean,4), "CV_Acc_Std": round(acc_std,4),
                       "CV_F1_Mean": round(f1_mean,4), "CV_F1_Std": round(f1_std,4)})

cv_df = pd.DataFrame(cv_records)
final_eval = eval_df[["Model","Test_Acc","Precision","Recall","F1_Score","ROC_AUC","Overfit_Gap"]].merge(cv_df, on="Model")
final_eval = final_eval.round(4)
print("\nConsolidated Evaluation Table:")
display(final_eval.set_index("Model"))

In [ ]:
metrics_plot  = ["Test_Acc", "Precision", "Recall", "F1_Score", "ROC_AUC"]
metric_labels = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]

fig, axes = plt.subplots(1, 5, figsize=(17, 5))
fig.suptitle("Figure 12: Test-Set Metric Comparison Across All Models", fontsize=13, fontweight="bold")

for ax, metric, label in zip(axes, metrics_plot, metric_labels):
    vals = [final_eval[final_eval["Model"]==m][metric].values[0] for m in MODEL_ORDER]
    bars = ax.bar(range(6), vals, color=COLORS_6, edgecolor="white", width=0.65)
    ax.set_title(label, fontsize=11)
    ax.set_xticks(range(6))
    ax.set_xticklabels(["LR","DT","RF","GB","SVM","KNN"], fontsize=9)
    ymin = max(0, min(vals)-0.05)
    ax.set_ylim(ymin, min(1.05, max(vals)+0.06))
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)

patches = [plt.Rectangle((0,0),1,1,color=COLORS_6[i]) for i in range(6)]
fig.legend(patches, MODEL_ORDER, loc="lower center", ncol=6, fontsize=9, bbox_to_anchor=(0.5,-0.08))
plt.tight_layout()
plt.show()

In [ ]:
x = np.arange(6)
w = 0.35
train_accs = [eval_df[eval_df["Model"]==m]["Train_Acc"].values[0] for m in MODEL_ORDER]
test_accs  = [eval_df[eval_df["Model"]==m]["Test_Acc"].values[0]  for m in MODEL_ORDER]

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, train_accs, w, label="Train Accuracy", color="#93C5FD", edgecolor="white")
b2 = ax.bar(x + w/2, test_accs,  w, label="Test Accuracy",  color=COLORS_6, edgecolor="white")
for bar, val in zip(list(b1)+list(b2), train_accs+test_accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f"{val:.3f}", ha="center", va="bottom", fontsize=8)
for i, (tr, te) in enumerate(zip(train_accs, test_accs)):
    if tr - te > 0.05:
        ax.annotate(f"gap={tr-te:.3f}",
                    xy=(i+w/2, te+0.005), xytext=(i+w/2, te+0.04),
                    arrowprops={"arrowstyle":"->","color":"#EF4444"},
                    ha="center", fontsize=8, color="#EF4444")
ax.set_xticks(x)
ax.set_xticklabels(MODEL_ORDER, rotation=12, ha="right")
ax.set_ylabel("Accuracy")
ax.set_ylim(0.70, 0.95)
ax.set_title("Figure 14: Train vs Test Accuracy (Overfitting Analysis)", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## Final Findings and Conclusion

In [ ]:
print("=" * 68)
print("  FINAL MODEL COMPARISON SUMMARY")
print("=" * 68)
display(final_eval.set_index("Model").style.highlight_max(
    subset=["Test_Acc","F1_Score","ROC_AUC","CV_F1_Mean"], color="#d1fae5"
).highlight_min(
    subset=["Overfit_Gap"], color="#fef9c3"
))

### Conclusions

**Dataset:** 10,000 student records with 10 features (8 numerical, 2 categorical binary). No missing values, no duplicates, no outliers.

**Top predictive features** (by consistent ranking across Random Forest, Gradient Boosting, and Permutation Importance):
1. AptitudeTestScore — strongest single numerical predictor
2. HSC_Marks — Class 12 performance 
3. ExtracurricularActivities — strongest binary categorical predictor

**Model results** (evaluated on an untouched 2,000-sample test set with 5-fold cross-validation):
- Logistic Regression achieved the highest test accuracy (80.85%) and ROC-AUC (0.8837) with no overfitting (train-test gap: −0.011).
- SVM was the most stable model with the smallest overfit gap (0.008) and the second-highest accuracy (80.05%).
- Random Forest and Gradient Boosting showed moderate overfitting; further hyperparameter tuning could improve their test performance.
- Decision Tree showed the highest overfitting (gap: 0.128) and the lowest test accuracy (75.00%).

**Important note:** Correlation between a feature and placement outcome does not imply causation. The model identifies statistical associations in this specific dataset.

**Future scope:** Hyperparameter tuning with GridSearchCV, SHAP explainability, larger and more diverse datasets, deployment as a web application.